In [383]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
import joblib

In [384]:
# load dataset
web = pd.read_csv('website_blocker_dataset.csv')
# data exploration
print(web.head())
print(web.tail())
print(web.index)
print(web.columns)
web.info()
print(web.describe())

                 Url   Labels
0  labaran batsa.com  Blocked
1         python.org  Allowed
2          harka.com  Blocked
3        physics.com  Allowed
4        gidan harka  Blocked
               Url   Labels
118  Tsuliya.com    Blocked
119   Gindi zallah  Blocked
120     bet9ja.com  Blocked
121       spot bet  Blocked
122    Agriculture  Allowed
RangeIndex(start=0, stop=123, step=1)
Index(['Url', 'Labels'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Url     123 non-null    object
 1   Labels  123 non-null    object
dtypes: object(2)
memory usage: 2.1+ KB
                Url   Labels
count           123      123
unique          122        2
top     lesbian.com  Allowed
freq              2       69


In [385]:
# data cleaning
web.isnull().sum()
web.dropna(inplace=True)
web.duplicated().sum()
web.drop_duplicates(inplace=True)

In [386]:
# data extraction
X = web['Url']
y = web['Labels']

In [387]:
# Test Alogorith performance
# Algorith performance
models = {
    "LogisticRegression": LogisticRegression(),
    "NaiveBayes": MultinomialNB(),
    "SVM": LinearSVC()
}

for name, model in models.items():

    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1,2),stop_words='english')),
        ("model", model)
    ])

    scores = cross_val_score(
        pipe,
        X,
        y,
        cv=3
    )

    print(name, scores.mean())

LogisticRegression 0.7465447154471544
NaiveBayes 0.8278455284552845
SVM 0.8278455284552845


In [388]:
# Choose algorith winner and train model with it
pipe = Pipeline([
    ('tfidf',TfidfVectorizer(ngram_range=(1,2),stop_words='english')),
    ('Model',LinearSVC())
])

In [389]:
X = web["Url"]
y = web["Labels"]
#  train test split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
# create model and train
pipe.fit(X_train,y_train)
print('Successfully')

Successfully


In [390]:
predictions = pipe.predict(X_test)
print(predictions)

['Allowed' 'Allowed' 'Allowed' 'Blocked' 'Blocked' 'Allowed' 'Allowed'
 'Allowed' 'Allowed' 'Allowed' 'Allowed' 'Allowed' 'Blocked' 'Blocked'
 'Blocked' 'Blocked' 'Blocked' 'Blocked' 'Allowed' 'Blocked' 'Allowed'
 'Blocked' 'Allowed' 'Blocked' 'Allowed']


In [391]:
# Accuracy score
score = accuracy_score(y_test,predictions)
print('Accuracy Score',score)
# check underfitting or overfitting
print('Train',pipe.score(X_train,y_train))
print('Test',pipe.score(X_test,y_test))

Accuracy Score 0.88
Train 1.0
Test 0.88


In [392]:
# confusion matrix
cmatrix = confusion_matrix(y_test,predictions)
print('Confusion Matrix',cmatrix)
# classification report
print(classification_report(y_test,predictions))

Confusion Matrix [[11  0]
 [ 3 11]]
              precision    recall  f1-score   support

     Allowed       0.79      1.00      0.88        11
     Blocked       1.00      0.79      0.88        14

    accuracy                           0.88        25
   macro avg       0.89      0.89      0.88        25
weighted avg       0.91      0.88      0.88        25



In [393]:
# cross validation score
cvalidation = cross_val_score(pipe,X,y,cv=3)
print('Cross Validations Score',cvalidation)

Cross Validations Score [0.85365854 0.80487805 0.825     ]


In [394]:
# new predictions
new_urls = [
    "free porn video",
    "jamb portal",
    "python tutorial",
    "casino bonus",
    "openai.com",
    'instagram.com',
    'facebook login',
    'gindi',
    'tsuliya dadi',
    'cin gindin matan hausawa',
    'bbc hausa news',
    'xan can iskanci gindi',
    'matan bariki',
    'lesbian hausawa'
]

results = pipe.predict(new_urls)
for url, result in zip(new_urls, results):
    print(url, "=>", result)

free porn video => Blocked
jamb portal => Allowed
python tutorial => Allowed
casino bonus => Allowed
openai.com => Allowed
instagram.com => Allowed
facebook login => Allowed
gindi => Blocked
tsuliya dadi => Blocked
cin gindin matan hausawa => Blocked
bbc hausa news => Allowed
xan can iskanci gindi => Blocked
matan bariki => Blocked
lesbian hausawa => Blocked


In [395]:
# save model
joblib.dump(pipe, "blocker_brain.pkl")

print("Model saved successfully")

Model saved successfully


In [396]:
# brain test ability
brain = joblib.load("blocker_brain.pkl")

In [400]:
# test 
tests = [
    "free porn video",
    "jamb portal",
    "python tutorial",
    "casino bonus",
    'xvideos.com',
    'labaran cin gindi',
    'batsa novels',
    'labaran duniya',
    'hausa novels',
    'maryam harka',
    'jamila harka',
    'cin gindi dadi',
    'www.computer networking',
    'student login details',
    'gindi',
    'Mathematics'
]

print(brain.predict(tests))

['Blocked' 'Allowed' 'Allowed' 'Allowed' 'Blocked' 'Blocked' 'Blocked'
 'Allowed' 'Allowed' 'Blocked' 'Blocked' 'Blocked' 'Allowed' 'Allowed'
 'Blocked' 'Allowed']
